In [ ]:
import pandas as pd

# Läs bara de första 10 raderna
df = pd.read_parquet("../data/fhvhv_tripdata_2020-01.parquet")

df = df.head(10)

print(df.shape)
print(df.dtypes)
df.head()




(10, 24)
hvfhs_license_num                  str
dispatching_base_num               str
originating_base_num               str
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int64
DOLocationID                     int64
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag                str
shared_match_flag                  str
access_a_ride_flag                 str
wav_request_flag                   str
wav_match_flag                     str
dtype: object
(19306090, 24)
hvfhs_license_num         

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2022-10-01 00:07:49,2022-10-01 00:10:31,2022-10-01 00:10:31,2022-10-01 00:21:47,235,174,3.65,...,1.24,0.0,0.0,0.00,12.39,N,N,,N,N
1,HV0003,B03404,B03404,2022-10-01 00:30:24,2022-10-01 00:33:54,2022-10-01 00:35:23,2022-10-01 00:47:21,241,254,2.33,...,1.03,0.0,0.0,0.00,11.23,N,N,,N,N
2,HV0003,B03404,B03404,2022-10-01 00:49:44,2022-10-01 00:52:50,2022-10-01 00:54:01,2022-10-01 01:08:20,174,250,5.69,...,1.59,0.0,0.0,0.00,16.38,N,N,,N,N
3,HV0003,B03404,B03404,2022-10-01 00:47:46,2022-10-01 00:48:48,2022-10-01 00:52:08,2022-10-01 01:14:31,211,265,6.05,...,0.00,0.0,0.0,0.00,32.25,N,N,,N,N
4,HV0003,B03404,B03404,2022-10-01 00:06:17,2022-10-01 00:06:38,2022-10-01 00:07:06,2022-10-01 00:13:54,44,44,1.56,...,0.71,0.0,0.0,0.02,8.26,N,N,,N,N


In [6]:
import pandas as pd

# Läs in en fil och kolla kolumnerna
df = pd.read_parquet("../data/fhvhv_tripdata_2020-01.parquet", engine="pyarrow")

print("Kolumner i datasetet:")
print(df.columns.tolist())

print("\nFinns PULocationID?", "PULocationID" in df.columns)
print("Finns DOLocationID?", "DOLocationID" in df.columns)

print("\nExempel på värden:")
print(df[["PULocationID", "DOLocationID"]].head(10))

Kolumner i datasetet:
['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']

Finns PULocationID? True
Finns DOLocationID? True

Exempel på värden:
   PULocationID  DOLocationID
0           148            90
1           114            79
2             4           125
3           231           113
4           114           144
5           144           137
6           249           148
7           148             4
8            79             7
9           140           236


In [7]:
import pandas as pd

# Läs in en liten del av tripdata
trips = pd.read_parquet("../data/fhvhv_tripdata_2020-01.parquet", engine="pyarrow").head(1000)

# Ladda ner zonfilen om den saknas
import requests
from pathlib import Path
zone_file = Path("../data/taxi_zone_lookup.csv")
if not zone_file.exists():
    r = requests.get("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
    zone_file.write_bytes(r.content)

zones = pd.read_csv(zone_file)

# Joina pickup-zon
df = trips.merge(
    zones[["LocationID", "Zone", "Borough"]].rename(columns={"LocationID": "PULocationID", "Zone": "pickup_zone", "Borough": "pickup_borough"}),
    on="PULocationID", how="left"
)

# Joina dropoff-zon
df = df.merge(
    zones[["LocationID", "Zone", "Borough"]].rename(columns={"LocationID": "DOLocationID", "Zone": "dropoff_zone", "Borough": "dropoff_borough"}),
    on="DOLocationID", how="left"
)

print("Kollar att zoner finns:")
print(df[["PULocationID", "pickup_zone", "pickup_borough", "DOLocationID", "dropoff_zone", "dropoff_borough"]].head(10))

print(f"\nNull-värden i pickup_zone: {df['pickup_zone'].isna().sum()}")
print(f"Null-värden i dropoff_zone: {df['dropoff_zone'].isna().sum()}")

Kollar att zoner finns:
   PULocationID              pickup_zone pickup_borough  DOLocationID  \
0           148          Lower East Side      Manhattan            90   
1           114  Greenwich Village South      Manhattan            79   
2             4            Alphabet City      Manhattan           125   
3           231     TriBeCa/Civic Center      Manhattan           113   
4           114  Greenwich Village South      Manhattan           144   
5           144      Little Italy/NoLiTa      Manhattan           137   
6           249             West Village      Manhattan           148   
7           148          Lower East Side      Manhattan             4   
8            79             East Village      Manhattan             7   
9           140          Lenox Hill East      Manhattan           236   

              dropoff_zone dropoff_borough  
0                 Flatiron       Manhattan  
1             East Village       Manhattan  
2                Hudson Sq       Manh

In [8]:
import pandas as pd
from pathlib import Path

# Läs in en liten del
trips = pd.read_parquet("../data/fhvhv_tripdata_2020-01.parquet", engine="pyarrow").head(10000)
zones = pd.read_csv("../data/taxi_zone_lookup.csv")

# Join
df = trips.merge(
    zones[["LocationID", "Zone", "Borough"]].rename(columns={"LocationID": "PULocationID", "Zone": "pickup_zone", "Borough": "pickup_borough"}),
    on="PULocationID", how="left"
)

# Aggregation: antal resor och snittpris per borough
agg = df.groupby("pickup_borough").agg(
    antal_resor=("PULocationID", "count"),
    snitt_pris=("base_passenger_fare", "mean")
).round(2).sort_values("antal_resor", ascending=False)

print(agg)

                antal_resor  snitt_pris
pickup_borough                         
Manhattan              4569       22.22
Brooklyn               2746       20.82
Queens                 1534       19.81
Bronx                  1078       17.62
Staten Island            72       22.40


In [9]:
import pandas as pd

# Läs in data (använder samma df från förra cellen)
trips = pd.read_parquet("../data/fhvhv_tripdata_2020-01.parquet", engine="pyarrow").head(10000)
zones = pd.read_csv("../data/taxi_zone_lookup.csv")

df = trips.merge(
    zones[["LocationID", "Zone", "Borough"]].rename(columns={"LocationID": "PULocationID", "Zone": "pickup_zone", "Borough": "pickup_borough"}),
    on="PULocationID", how="left"
)

# Räkna resor per borough + zon
zone_counts = df.groupby(["pickup_borough", "pickup_zone"]).size().reset_index(name="antal_resor")

# Window function: ranka zoner inom varje borough
zone_counts["rank"] = zone_counts.groupby("pickup_borough")["antal_resor"].rank(method="min", ascending=False).astype(int)

# Visa topp 3 per borough
topp3 = zone_counts[zone_counts["rank"] <= 3].sort_values(["pickup_borough", "rank"])
print(topp3.to_string(index=False))

pickup_borough                    pickup_zone  antal_resor  rank
         Bronx         Mott Haven/Port Morris           61     1
         Bronx                   East Tremont           57     2
         Bronx          Van Cortlandt Village           54     3
      Brooklyn      Williamsburg (North Side)          187     1
      Brooklyn                     Park Slope          179     2
      Brooklyn      Williamsburg (South Side)          151     3
     Manhattan                   East Village          300     1
     Manhattan                Lower East Side          205     2
     Manhattan                   West Village          151     3
        Queens                        Astoria          133     1
        Queens Long Island City/Hunters Point          115     2
        Queens                Jackson Heights           91     3
 Staten Island                    Great Kills            9     1
 Staten Island        Bloomfield/Emerson Hill            7     2
 Staten Island      Saint

In [5]:
df1 = pd.read_parquet("../data/fhvhv_tripdata_2022-10.parquet")
df = df.head(10)

In [4]:
print(df1.shape)
print(df1.dtypes)
df1.head()


(19306090, 24)
hvfhs_license_num                  str
dispatching_base_num               str
originating_base_num               str
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int64
DOLocationID                     int64
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag                str
shared_match_flag                  str
access_a_ride_flag                 str
wav_request_flag                   str
wav_match_flag                     str
dtype: object


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2022-10-01 00:07:49,2022-10-01 00:10:31,2022-10-01 00:10:31,2022-10-01 00:21:47,235,174,3.65,...,1.24,0.0,0.0,0.00,12.39,N,N,,N,N
1,HV0003,B03404,B03404,2022-10-01 00:30:24,2022-10-01 00:33:54,2022-10-01 00:35:23,2022-10-01 00:47:21,241,254,2.33,...,1.03,0.0,0.0,0.00,11.23,N,N,,N,N
2,HV0003,B03404,B03404,2022-10-01 00:49:44,2022-10-01 00:52:50,2022-10-01 00:54:01,2022-10-01 01:08:20,174,250,5.69,...,1.59,0.0,0.0,0.00,16.38,N,N,,N,N
3,HV0003,B03404,B03404,2022-10-01 00:47:46,2022-10-01 00:48:48,2022-10-01 00:52:08,2022-10-01 01:14:31,211,265,6.05,...,0.00,0.0,0.0,0.00,32.25,N,N,,N,N
4,HV0003,B03404,B03404,2022-10-01 00:06:17,2022-10-01 00:06:38,2022-10-01 00:07:06,2022-10-01 00:13:54,44,44,1.56,...,0.71,0.0,0.0,0.02,8.26,N,N,,N,N
